#CONSUMER BUYING BEHAVIOUR ANALYSIS


#Problem Statement

Customer buying behaviour Analysis is a detailed analysis of a company’s ideal customers. It helps a business to better understand its customers and makes it easier for them to modify products according to the specific needs, behaviors and concerns of different types of customers.

This analysis helps a business to modify its product based on its target customers from different types of customer segments. For example, instead of spending money to market a new product to every customer in the company’s database, a company can analyze which customer segment is most likely to buy the product and then market the product only on that particular segment.



## IMPORTING LIBRARIES

In [ ]:
import numpy as np                               
import pandas as pd                               
import seaborn as sns
import plotly.graph_objects as go
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

##Let's first start with exploring the features in the dataset. Based on the data description, the features can be grouped into infomation about customers, information about the products, promotions, and the place where the purchases were made through.

Attributes

People

ID: Customer's unique identifier

Year_Birth: Customer's birth year

Education: Customer's education level

Marital_Status: Customer's marital status

Income: Customer's yearly household income

Kidhome: Number of children in customer's household

Teenhome: Number of teenagers in customer's household

Dt_Customer: Date of customer's enrollment with the company

Recency: Number of days since customer's last purchase

Complain: 1 if customer complained in the last 2 years, 0 otherwise

Products

MntWines: Amount spent on wine in last 2 years

MntFruits: Amount spent on fruits in last 2 years

MntMeatProducts: Amount spent on meat in last 2 years

MntFishProducts: Amount spent on fish in last 2 years

MntSweetProducts: Amount spent on sweets in last 2 years

MntGoldProds: Amount spent on gold in last 2 year

Promotion

NumDealsPurchases: Number of purchases made with a discount

AcceptedCmp1: 1 if customer accepted the offer in the 1st campaign, 0 otherwise

AcceptedCmp2: 1 if customer accepted the offer in the 2nd campaign, 0 otherwise

AcceptedCmp3: 1 if customer accepted the offer in the 3rd campaign, 0 otherwise

AcceptedCmp4: 1 if customer accepted the offer in the 4th campaign, 0 otherwise

AcceptedCmp5: 1 if customer accepted the offer in the 5th campaign, 0 otherwise

Response: 1 if customer accepted the offer in the last campaign, 0 otherwise

Place

NumWebPurchases: Number of purchases made through the company’s web site

NumCatalogPurchases: Number of purchases made using a catalogue

NumStorePurchases: Number of purchases made directly in stores

NumWebVisitsMonth: Number of visits to company’s web site in the last month

Target

Need to perform clustering & Apriori to summarize customer segments.

#IMPORTING DATASET

In [ ]:
# Import the Dataset
df = pd.read_csv("../input/customer-personality-analysis/marketing_campaign.csv", sep="\t")
df.head()

In [ ]:
df.info()
# observation -> here, total 2240 samples and 29 attributes

#Let's look at the summary statistics of the data. We can see that the maximum income is much higher than the income at the 3rd quatile (almost 10 times greater). Also, the maximum amount spent on wines and meat products (i.e. MntWines, MntMeatProduct) are significantly greater than the one spent on other products.

In [ ]:
df.describe()

#INSPECTING DATAFRAME

In [ ]:
df.shape

#Checking null values

### observation -> data['Income'] have 24 null values

In [ ]:
df.isna().sum()

#DATA CLEANING
## we can see that two columns 'Z_CostContact' and 'Z_Revenue' which haven't been described by the data.

### Dropping columns because they will not contribute anything in model building

In [ ]:
df = df.drop(['Z_CostContact', 'Z_Revenue'],axis=1)
df.head()

In [ ]:
plt.rcParams.update(plt.rcParamsDefault)

# Checking correlation between the attributes

In [ ]:
plt.figure(figsize=(18,12))
sns.heatmap(df.corr(), annot=True)
plt.show()

####No two columns are too much correlated with each other so we can't drop any column on the basis of correlation.

### Checking for correlation by unstacking data

##It is used to calculate how one variable is correlated/ dependent on other variable.
Extreme values signify high correlation.
Multicollinear variables with correlation more than a threshold are usually dropped from the dataset.

In [ ]:
# Check the Correlation Report
corr_data = df.corr()
corr_data.abs().unstack().sort_values(ascending=False)[24:50:2]

#PREPROCESSING OF THE DATASET

### Filling the missing value in the income my mean

In [ ]:
df['Income'] = df['Income'].fillna(df['Income'].mean())
df.isna().any() 

In [ ]:
df.head()

# Checking number of unique categories present in the "Marital_Status"

In [ ]:
df['Marital_Status'].value_counts()  

In [ ]:
sns.countplot(df.Marital_Status)
sns.set(rc={'figure.figsize':(4,4)})
plt.show()

In [ ]:
df['Marital_Status'] = df['Marital_Status'].replace(['Married', 'Together'],'relationship')
df['Marital_Status'] = df['Marital_Status'].replace(['Divorced', 'Widow', 'Alone', 'YOLO', 'Absurd'],'Single')

##In the above cell we are grouping 'Married', 'Together' as "relationship"
##Whereas 'Divorced', 'Widow', 'Alone', 'YOLO', 'Absurd' as "Single"

## Count of different values present in Marital_Status

In [ ]:
df['Marital_Status'].value_counts()  

In [ ]:
sns.countplot(df.Marital_Status)
sns.set(rc={'figure.figsize':(4,4)})
plt.show()

#Separating Products to different Dataframe for Association Rule Mining

In [ ]:
product_data = []
for i in range(0, len(df)):
  productdata = [df['MntWines'][i], df['MntFruits'][i], 
                  df['MntMeatProducts'][i], df['MntFishProducts'][i], 
                  df['MntSweetProducts'][i], df['MntGoldProds'][i]]
  product_data.append(productdata)
Products_DF = pd.DataFrame(product_data, columns = ['Wines', 'Fruits', 'Meat', 'Fish', 'Sweets', 'Gold'])
Products_DF.head()

##Combining different dataframe into a single column to reduce the number of dimension

In [ ]:
df['Kids'] = df['Kidhome'] + df['Teenhome']
df['Expenses'] = df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] + df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds']
df['TotalAcceptedCmp'] = df['AcceptedCmp1'] + df['AcceptedCmp2'] + df['AcceptedCmp3'] + df['AcceptedCmp4'] + df['AcceptedCmp5'] + df['Response']
df['NumTotalPurchases'] = df['NumWebPurchases'] + df['NumCatalogPurchases'] + df['NumStorePurchases'] + df['NumDealsPurchases']

##Deleting some column to reduce dimension and complexity of model

In [ ]:
col_del = ["AcceptedCmp1" , "AcceptedCmp2", "AcceptedCmp3" , "AcceptedCmp4","AcceptedCmp5", "Response","NumWebVisitsMonth", "NumWebPurchases","NumCatalogPurchases","NumStorePurchases","NumDealsPurchases" , "Kidhome", "Teenhome","MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
df=df.drop(columns=col_del,axis=1)
df.head()

### Adding a column "Age" in the dataframe

In [ ]:
df['Age'] = 2015 - df["Year_Birth"]

#Count of different values present in Education

In [ ]:
df['Education'].value_counts()

# Changing category into UG and PG only

In [ ]:
df['Education'] = df['Education'].replace(['PhD','2n Cycle','Graduation', 'Master'],'PG')  
df['Education'] = df['Education'].replace(['Basic'], 'UG')

## Number of days a customer was engaged with company

# Changing Dt_customer into timestamp format

In [ ]:
df['Dt_Customer'] = pd.to_datetime(df.Dt_Customer)
df['first_day'] = '01-01-2015'
df['first_day'] = pd.to_datetime(df.first_day)
df['day_engaged'] = (df['first_day'] - df['Dt_Customer']).dt.days

In [ ]:
df=df.drop(columns=["ID", "Dt_Customer", "first_day", "Year_Birth", "Dt_Customer", "Recency", "Complain"],axis=1)
df.shape

In [ ]:
df.head()

#**VISUALIZATION**

#Then, let's look at the categorical distributions of education level and the marital status of the customers. We can see that majority of the customers have a graduation degree, and most of them are in relationship or married

In [ ]:
plt.rcParams.update(plt.rcParamsDefault)

#ANALYSIS OF THE CORRELATION BETWEEN MARITAL STATUS AND EXPENSES WITH RESPECT TO EDUCATION

In [ ]:
plt.figure(figsize=(8,8))
sns.barplot(x=df['Marital_Status'], y=df['Expenses'], hue = df["Education"])
plt.title("Analysis of the Correlation between Marital Status and Expenses with respect to Education")
plt.show()

##Observation: Less number of single customers and very high expenses for single customers.

#ANALYSIS OF THE CORRELATION BETWEEN MARITAL STATUS AND EXPENSES

In [ ]:
plt.figure(figsize=(8,8))
sns.barplot(x=df['Marital_Status'], y=df['Expenses'])
plt.title("Analysis of the Correlation between Marital Status and Expenses")
plt.show()

#DISTRIBUTION OF EXPENSES WITH RESPECT TO MARITAL STATUS

In [ ]:
plt.figure(figsize=(8,8))
plt.hist("Expenses", data = df[df["Marital_Status"] == "relationship"], alpha = 0.5, label = "relationship")
plt.hist("Expenses", data = df[df["Marital_Status"] == "Single"], alpha = 0.5, label = "Single")
plt.title("Distribution of Expenses with respect to Marital Status")
plt.xlabel("Expenses")
plt.legend(title = "Marital Status")
plt.show()

#DISTRIBUTION OF EXPENSES WITH RESPECT TO EDUCATION

In [ ]:
#from numpy.core.fromnumeric import size
plt.figure(figsize=(8,8))
plt.hist("Expenses", data = df[df["Education"] == "PG"], alpha = 0.5, label = "PG")
plt.hist("Expenses", data = df[df["Education"] == "UG"], alpha = 0.5, label = "UG")
plt.title("Distribution of Expenses with respect to Education")
plt.xlabel("Expenses")
plt.legend(title = "Education")
plt.show()

#DISTRIBUTION OF NUMBER OF TOTAL EXPENSES WITH RESPECT TO EDUCATION

In [ ]:
plt.figure(figsize=(8,8))
plt.hist("NumTotalPurchases", data = df[df["Education"] == "PG"], alpha = 0.5, label = "PG")
plt.hist("NumTotalPurchases", data = df[df["Education"] == "UG"], alpha = 0.5, label = "UG")
plt.title("Distribution of Number of Total Purchases with respect to Education")
plt.xlabel("Number of Total Purchases")
plt.legend(title = "Education")
plt.show()

#DISTRIBUTION OF AGE WITH RESPECT TO MARITAL STATUS

In [ ]:
plt.figure(figsize=(8,8))
plt.hist("Age", data = df[df["Marital_Status"] == "relationship"], alpha = 0.5, label = "relationship")
plt.hist("Age", data = df[df["Marital_Status"] == "Single"], alpha = 0.5, label = "Single")
plt.title("Distribution of Age with respect to Marital Status")
plt.xlabel("Age")
plt.legend(title = "Marital Status")
plt.show()

#DISTRIBUTION OF INCOME WITH RESPECT TO MARITAL STATUS

In [ ]:
plt.figure(figsize=(8,8))
plt.hist("Income", data = df[df["Marital_Status"] == "relationship"], alpha = 0.5, label = "relationship")
plt.hist("Income", data = df[df["Marital_Status"] == "Single"], alpha = 0.5, label = "Single")
plt.title("Distribution of Income with respect to Marital Status")
plt.xlabel("Income")
plt.legend(title = "Marital Status")
plt.show()

#ANALYSIS OF THE DISTRIBUTION OF PEOPLE ACCORDING TO MARITAL STATUS

In [ ]:
plt.figure(figsize=(8,8))
plt.pie(df["Marital_Status"].value_counts(), labels = ["relationship", "Single"], autopct='%1.1f%%', counterclock=False)
plt.legend()
plt.show()

##35% of the customer are single whereas more 64% are in relationship.

#ANALYSIS OF THE DISTRIBUTION OF PEOPLE ACCORDING TO EDUCATION

In [ ]:
plt.figure(figsize=(8,8))
plt.pie(df["Education"].value_counts(), labels = ["PG", "UG"], autopct='%1.1f%%', counterclock=False)
plt.legend()
plt.show()

##More than 97% customer are from PG background. and Approx. 2% are from UG.

#DISTRIBUTION OF EXPENSES BASED ON EDUCATION

In [ ]:
sns.barplot(x = df['Expenses'],y = df['Education']);
plt.title('Total Expense based on the Education Level');
plt.show()

#INCOME BASED ON EDUCATION LEVEL

In [ ]:
sns.barplot(x = df['Income'],y = df['Education']);
plt.title('Total Income based on the Education Level');
plt.show()

In [ ]:
df.describe()

##LABEL ENCODING

In [ ]:
cate = []
for i in df.columns:
    if (df[i].dtypes == "object"):
        cate.append(i)

print(cate)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn import preprocessing 

In [ ]:
lbl_encode = LabelEncoder()
for i in cate:
    df[i]=df[[i]].apply(lbl_encode.fit_transform)

In [ ]:
df1 = df.copy()

In [ ]:
df1.head(3)

#OUTLIERS DETECTION

In [ ]:
# check for outliers
plt.figure(figsize=(5,5))
ax = sns.boxplot(data=df1 , orient="h")
plt.title('A boxplot: Outliers in the dataset', color = 'blue')
plt.xlabel('Count/ Frequency')
plt.show()

In [ ]:
from math import sqrt
# Drop Outliers
q3 = df1.quantile(0.75)
q1 = df1.quantile(0.25)
iqr = q3-q1
lower_range = q1 - (1.5 * iqr)
upper_range = q3 + (1.5 * iqr)

df1 = df1[~( (df1 < lower_range)|(df1 > upper_range) ).any(axis=1)]

#**K-MEANS CLUSTERING**

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
wcss=[] ##Initializing the list for the values of WCSS  
for i in range (1,11): #for diferent values of k ranging from 1 to 10
 kmeans=KMeans(n_clusters=i,init='k-means++',random_state=42)
 kmeans.fit(df1)
 wcss.append(kmeans.inertia_)
plt.figure(figsize=(16,8))
plt.plot(range(1,11),wcss, 'bx-')
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

##As it is not very clear from the elbow method that which value of K to choose.


##Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score 

In [ ]:
silhouette_scores = []
for i in range(2,10):
    m1=KMeans(n_clusters=i, random_state=42)
    c = m1.fit_predict(df1)
    silhouette_scores.append(silhouette_score(df1, m1.fit_predict(df1))) 
plt.bar(range(2,10), silhouette_scores) 
plt.xlabel('Number of clusters', fontsize = 20) 
plt.ylabel('S(i)', fontsize = 20) 
plt.show()

##Here we are using Silhouette score to measure the value of K

In [ ]:
silhouette_scores

In [ ]:
# Getting the maximum value of silhouette score and adding 2 in index because index starts from 2.
sc=max(silhouette_scores)
number_of_clusters=silhouette_scores.index(sc)+2
print("Number of Cluster Required is : ", number_of_clusters)

#Model Building


In [ ]:
# Training a predicting using K-Means Algorithm.

kmeans=KMeans(n_clusters=number_of_clusters, random_state=42).fit(df1)
pred=kmeans.predict(df1)


# Appending those cluster value into main dataframe (without standard-scalar)

df1['cluster'] = pred + 1

In [ ]:
df1.head()

##Clustering

In [ ]:
df1['cluster'].value_counts()

In [ ]:
pl = sns.countplot(x=df1["cluster"])
pl.set_title("Distribution Of The Clusters")
plt.show()

##OBSERVATIONS :-

As we can see here that weightage of customer are more in cluster 1 as compare to other.

In [ ]:
# Clusters interpretation 
sns.set(rc={'axes.facecolor':'black', 'figure.facecolor':'black', 'axes.grid' : False, 'font.family': 'Ubuntu'})

for i in df1:
    diag = sns.FacetGrid(df1, col = "cluster", hue = "cluster", palette = "Set1")
    diag.map(plt.hist, i, bins=6, ec="k") 
    diag.set_xticklabels(rotation=25, color = 'white')
    diag.set_yticklabels(color = 'white')
    diag.set_xlabels(size=16, color = 'white')
    diag.set_titles(size=16, color = '#f01132', fontweight="bold")
    diag.fig.set_figheight(6)

plt.show()

##OBSERVATIONS: Based on above information we can divide customer into 2 parts:-
Highly Active Customer :- These customers belong to cluster one.

Least Active Customer :- These customers belong to cluster two.

##1.Characteristics of Highly Active Customer
##In terms of Education
Highly Active Customer are from PG background

##In terms of Marital_status
Number of people in relationship are approx. two times of single people

##In terms of Income
Income of Highly active customer are little less as compare to least active customer.

##In terms of Kids
Highly active customer have more number of children as compare to other customer ( avg. of 1 child ).

##In terms of Expenses
Expenses of Highly Active customer are less as compare to least.
These customer spent avg. of approx. 100-200 unit money.

##In terms of Age
Age of these customer are between 25 to 75.
Maximum customer age are between 40 to 50.

##In terms of day_engaged
Highly Active customer are more loyal as they engaged with company for longer period of time.


##2.Characteristics of Least Active Customer
##In terms of Education
Least Active Customer are from UG backgroud

##In terms of Marital_status
Number of people in relationship are approx. equal to single people

##In terms of Income
Income of Least active customer are very less or say negligible.

##In terms of Kids
Only few of these customer have child.

##In terms of Expenses
Expenses of Least Active customer are very less or say negligible.

##In terms of Age
Age of these customer are between 15 to 30.

##In terms of day_engaged
Least Active customer are not much enrolled with company for longer time

In [ ]:
PLOT = go.Figure()
for C in list(df1.cluster.unique()):
    

    PLOT.add_trace(go.Scatter3d(x = df1[df1.cluster == C]['Income'],
                                y = df1[df1.cluster == C]['Age'],
                                z = df1[df1.cluster == C]['day_engaged'],                        
                                mode = 'markers',marker_size = 6, marker_line_width = 1,
                                name = str(C)))
PLOT.update_traces(hovertemplate='Income: %{x} <br>Age: %{y} <br>Days Engaged: %{z}')

    
PLOT.update_layout(width = 800, height = 800, autosize = True, showlegend = True,
                   scene = dict(xaxis=dict(title = 'Income', titlefont_color = 'black'),
                                yaxis=dict(title = 'Age', titlefont_color = 'black'),
                                zaxis=dict(title = 'Days Engaged', titlefont_color = 'black')),
                   font = dict(family = "Gilroy", color  = 'black', size = 12))

#MODEL EVALUATION

In [ ]:
#kmeans

from sklearn.metrics import confusion_matrix,classification_report
print("ConfusionMatrix \n",confusion_matrix(kmeans.labels_, pred))
print("classification report \n", classification_report(kmeans.labels_, pred))

#**ASSOCIATION RULE MINING: APRIORI ALGORITHM**

Association Rule Mining is used when we want to find an association between different objects in a set, find frequent patterns in a transaction database. 

The Apriori algorithm is the simplest technique to identify the underlying relationships between different types of elements.

Here we use this algorithm to find out which customers are best suited for a given item. Thereby helping businesses promote the right target customers to increase efficiency and save costs.

In [ ]:
pip install apyori

In [ ]:
from apyori import apriori
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

#Data Preparation for Apriori Algorithm

In [ ]:
data = df1.copy()
data.head()

In [ ]:
#Create Age segment
cut_labels_Age = ['Young', 'Adult', 'Mature', 'Senior']
cut_bins = [0, 30, 45, 65, 120]
data['Age_group'] = pd.cut(data['Age'], bins=cut_bins, labels=cut_labels_Age)
#Create Income segment
cut_labels_Income = ['Low income', 'Low to medium income', 'Medium to high income', 'High income']
data['Income_group'] = pd.qcut(data['Income'], q=4, labels=cut_labels_Income)
#Create day engaged segment
cut_labels_dayengaged = ['New customers', 'Discovering customers', 'Experienced customers', 'Old customers']
data['dayengaged_group'] = pd.qcut(data['day_engaged'], q=4, labels=cut_labels_dayengaged)
data=data.drop(columns=['Age','Income','day_engaged'])

Defining new segments according to the spending of customers on each product which will be based on:

Cluster 1 - Highly Active Customer

Cluster 2 - Least Active Customer


In [ ]:
cut_labels = ['Least Active Customer', 'Highly Active Customer']
data['Wines_segment'] = pd.qcut(Products_DF['Wines'][Products_DF['Wines']>0],q=[0, 0.5 ,1], labels=cut_labels).astype("object")
data['Fruits_segment'] = pd.qcut(Products_DF['Fruits'][Products_DF['Fruits']>0],q=[0, 0.5, 1], labels=cut_labels).astype("object")
data['Meat_segment'] = pd.qcut(Products_DF['Meat'][Products_DF['Meat']>0],q=[0, 0.5,1], labels=cut_labels).astype("object")
data['Fish_segment'] = pd.qcut(Products_DF['Fish'][Products_DF['Fish']>0],q=[0, 0.5, 1], labels=cut_labels).astype("object")
data['Sweets_segment'] = pd.qcut(Products_DF['Sweets'][Products_DF['Sweets']>0],q=[0, 0.5, 1], labels=cut_labels).astype("object")
data['Gold_segment'] = pd.qcut(Products_DF['Gold'][Products_DF['Gold']>0],q=[0, 0.5, 1], labels=cut_labels).astype("object")
data.replace(np.nan, "Inactive Customer",inplace=True)
data = data.astype(object)

In [ ]:
data.head()

#Applying Apriori Algorithm

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 999)
pd.options.display.float_format = "{:.3f}".format

In [ ]:
association = data.copy() 
association.head()

In [ ]:
association.drop(["Education", "Marital_Status", "Kids", "Expenses", "TotalAcceptedCmp", "NumTotalPurchases", "cluster"], axis = 1, inplace = True)
association.head()

#Setting the Association Rules

In [ ]:
df_ap = pd.get_dummies(association)
min_support = 0.08
max_len = 10
frequent_items = apriori(df_ap, use_colnames=True, min_support=min_support, max_len=max_len + 1)
rules = association_rules(frequent_items, metric='lift', min_threshold=1)

#Finding the "Highly Active Customers" when it comes to "Wine"

In [ ]:
product='Wines'
segment='Highly Active Customer'
target = '{\'%s_segment_%s\'}' %(product,segment)
results_personnel_care = rules[rules['consequents'].astype(str).str.contains(target, na=False)].sort_values(by='confidence', ascending=False)
results_personnel_care.head()

#Finding the "Highly Active Customers" when it comes to "Fruits"

In [ ]:
product='Fruits'
segment='Highly Active Customer'
target = '{\'%s_segment_%s\'}' %(product,segment)
results_personnel_care = rules[rules['consequents'].astype(str).str.contains(target, na=False)].sort_values(by='confidence', ascending=False)
results_personnel_care.head()

#Finding the "Highly Active Customers" when it comes to "Meat"

In [ ]:
product='Meat'
segment='Highly Active Customer'
target = '{\'%s_segment_%s\'}' %(product,segment)
results_personnel_care = rules[rules['consequents'].astype(str).str.contains(target, na=False)].sort_values(by='confidence', ascending=False)
results_personnel_care.head()

#Finding the "Highly Active Customers" when it comes to "Fish"

In [ ]:
product='Fish'
segment='Highly Active Customer'
target = '{\'%s_segment_%s\'}' %(product,segment)
results_personnel_care = rules[rules['consequents'].astype(str).str.contains(target, na=False)].sort_values(by='confidence', ascending=False)
results_personnel_care.head()

#Finding the "Highly Active Customers" when it comes to "Sweets"

In [ ]:
product='Sweets'
segment='Highly Active Customer'
target = '{\'%s_segment_%s\'}' %(product,segment)
results_personnel_care = rules[rules['consequents'].astype(str).str.contains(target, na=False)].sort_values(by='confidence', ascending=False)
results_personnel_care.head()

#Finding the "Highly Active Customers" when it comes to "Gold"

In [ ]:
product='Gold'
segment='Highly Active Customer'
target = '{\'%s_segment_%s\'}' %(product,segment)
results_personnel_care = rules[rules['consequents'].astype(str).str.contains(target, na=False)].sort_values(by='confidence', ascending=False)
results_personnel_care.head()

#Objective of Association Rule Mining

In this way, if we can find the Biggest Consumers for a particular product, multiple ways of market these products can be narrowed down to these customers and their needs. 

Thus, Apriori Algorithm will help the businesses to plan appropriate Product Pricings, making better decisions on Product Positioning and helping the business owner to work on product assortment and availability of different category of products, all based on the needs of these Highly Active Customers.

#**CLASSIFICATION: LOGISTIC REGRESSION**
To determine whether customers will purchase the company’s product or not?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
df2=df1.copy()
x = df2.drop('cluster', axis=1)
y = df2['cluster']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)

In [ ]:
#scaling
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [ ]:
log_reg = LogisticRegression()
log_reg.fit(x_train, y_train)

In [ ]:
y_predicted = log_reg.predict(x_test)

#MODEL EVALUATION

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score,confusion_matrix,f1_score

In [ ]:
#performance of model
print("Classification Report: \n", classification_report(y_test,y_predicted))
print("-" * 100)
print()
    
acc = accuracy_score(y_test, y_predicted)

print("Accuracy Score: ", acc)
print("-" * 100)
print()

f1 = f1_score(y_test, y_predicted)

print("F1 Score: ", f1)
print("-" * 100)
print()
    
print("Confusion Matrix: ")
plt.figure(figsize=(10, 5))
sns.heatmap(confusion_matrix(y_test, y_predicted), annot=True, fmt='g');
plt.title('Confusion Matrix', fontsize=20)
plt.show()

Observation: From, the above confusion matrix, it can be seen that Logistic Regression predicted only 3 incorrect values but rest all values were predicted correctly that means that the Logistic Regression classifier did perform reasonably well.
